# Supplementary Figs. 28-29 - benchmarking the longitudinal phased mCA caller on simulated data

Characterisation of the **longitudinal phased** mCA caller on simulated samples, followed by the
false-positive-rate analyses.

| section | what it establishes | writes |
|---|---|---|
| **High-cell-fraction sample phasing** | sensitivity, with phasing derived from a high-cell-fraction reference timepoint exactly as done for the cohort | `sensitivity_results_all_ref_cfs.csv` (Supp Fig 28) |
| **Within-family false positive rate** | the null, using other controls as pseudo-timepoints for a control that has no mCA | `Within_family_fpr_results.csv` (Supp Fig 29) |

This notebook **writes the result tables that
Supplementary Figs. 28 and 29 plot**
(`sensitivity_results_all_ref_cfs.csv` and `Within_family_fpr_results.csv`); those two notebooks read the
saved tables rather than re-running the pipelines.

The same caller is applied to the real cohort samples in
`Supplementary_Fig_31-38-mCA_calling_phased_timepoints.ipynb`, which produces the lower panels of
Supplementary Figs. 31-38. The shared configuration and function cells appear in both notebooks so each
runs independently.

## Data availability

The simulated samples are generated by spiking mCAs of known type, size and cell fraction into control
samples; the generating code is `Supplementary_Figs_26-30_Simulating_mCA_samples.ipynb` and
the control-selection step within it is Supplementary Fig. 30.

The simulated SNP files are derived from **real control samples**, so they carry those participants'
germline genotypes and are **not distributed with this code**. The control samples they are built from are
deposited under controlled access in the European Genome-phenome Archive and released to approved
researchers via a Data Access Committee.

| file | used for | location |
|---|---|---|
| `Simulated_samples_TEST_set/`, `..._TRAINING_set/` SNP files | the simulated samples the caller is run over | not distributed; derived from EGA-deposited controls |
| `simulation_manifest.csv` | ground truth and the seed for each simulated sample | written by `Supplementary_Figs_26-30_Simulating_mCA_samples.ipynb` |
| `truth_comparison.csv` | the **unphased** caller's calls matched against that ground truth, used here only for the unphased-vs-phased comparison | written by `Supplementary_Fig_26_mCA_caller_on_simulated_samples.py` in this repository, which runs the unphased caller over the simulated samples |
| `TWIST_CNV_panel_TE-95031423_h19.bed` | panel design, for null-region selection in the FPR analysis | Data_files |
| `chromosome_ideogram_hg19.txt` | chromosome ideograms | Data_files |

The summary tables this notebook writes are what Supplementary Figs. 28 and 29 plot, and those are
included in `Data_files/mCA_calling/Simulated_data/mCA_simulation_test_set/`.

## Configuration and shared functions


In [49]:
import numpy as np
import pandas as pd
import matplotlib
try:
except ImportError:
    BrokenBarHCollection = None
import os, warnings
from collections import defaultdict
from scipy.stats import ttest_1samp, t as t_dist

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [50]:
# === COLOR DEFINITIONS ===
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33); c2 = (1.00, 0.23, 0.19)
c3 = (1.00, 0.58, 0.00); c4 = (1.00, 0.80, 0.00)
c5 = (0.30, 0.85, 0.39); c6 = (0.35, 0.78, 0.98)
c7 = (0.20, 0.67, 0.86); c8 = (0.00, 0.48, 1.00)
c9 = (0.35, 0.34, 0.84); c10 = (0.00, 0.31, 0.57)

orange1='#feedde'; orange2='#fdbe85'; orange3='#fd8d3c'; orange4='#e6550d'; orange5='#a63603'
blue1='#eff3ff'; blue2='#bdd7e7'; blue3='#6baed6'; blue4='#3182bd'; blue5='#08519c'
green1='#edf8e9'; green2='#bae4b3'; green3='#74c476'; green4='#31a354'; green5='#006d2c'
grey1='#f7f7f7'; grey2='#cccccc'; grey3='#969696'; grey4='#636363'; grey5='#252525'
purple1='#f2f0f7'; purple2='#cbc9e2'; purple3='#9e9ac8'; purple4='#756bb1'; purple5='#54278f'
red1='#fee5d9'; red2='#fcae91'; red3='#fb6a4a'; red4='#de2d26'; red5='#a50f15'

In [51]:
ideogram_file = 'Data_files/chromosome_ideogram_hg19.txt'

chromosome_sizes = {
    'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276,
    'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022,
    'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895,
    'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753,
    'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520,
    'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560
}


In [52]:
def load_centromeres(filepath):
    """
    Parses the UCSC cytoband file to find the start/end of centromeres ('acen').
    Returns a dictionary: {'1': (start, end), '2': (start, end), ...}
    """
    
    df = pd.read_csv(filepath, sep="\t", comment='#', header=None, 
                     names=['chrom', 'start', 'end', 'name', 'type'])
    
    # Filter for centromeric regions ('acen')
    acen = df[df['type'] == 'acen'].copy()
    
    centromeres = {}
    for chrom, grp in acen.groupby('chrom'):
        # Centromeres usually span two bands (p-arm end, q-arm start)
        # We take the overall min start and max end.
        start = grp['start'].min()
        end = grp['end'].max()
        
        centromeres[chrom] = {'start': start, 'end': end}
        
    return centromeres

centromere_dict = load_centromeres('Data_files/chromosome_ideogram_hg19.txt')
print("Loaded centromeres for:", list(centromere_dict.keys()))

Loaded centromeres for: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']


In [53]:
# ── Where the simulated samples live ──────────────────────────────────────────
# The simulated test set is ~100 GB and is derived from real control samples, so it
# carries their germline genotypes and is NOT distributed with this code. Point
# SIMULATED_BASE at your own copy to run this notebook; everything below derives
# from it. See the data-availability note at the top.
SIMULATED_BASE = 'Data_files/mCA_calling/Simulated_data'

RESULTS_DIR   = SIMULATED_BASE + '/mCA_caller_simulation_results_TEST_set_v13_caller_all_cell_fractions'
SIMULATED_DIR = SIMULATED_BASE + '/Simulated_samples_TEST_set'
MANIFEST_CSV  = SIMULATED_DIR + '/simulation_manifest.csv'
OUTPUT_DIR    = RESULTS_DIR

# truth_comparison.csv is small and non-identifying, so it ships: read the staged copy
# if the full results tree is not present.
TRUTH_CSV = RESULTS_DIR + '/truth_comparison.csv'
if not os.path.exists(TRUTH_CSV):
    TRUTH_CSV = SIMULATED_BASE + '/mCA_simulation_test_set/truth_comparison.csv'

# Fail early and clearly rather than deep inside a pipeline function
_missing = [p for p in (SIMULATED_DIR, MANIFEST_CSV, TRUTH_CSV) if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError(
        "This notebook needs the simulated test set, which is not distributed with the code "
        "(see the data-availability note at the top). Missing:\n  " + "\n  ".join(_missing) +
        "\nSet SIMULATED_BASE above to your own copy of the simulated samples."
    )

# Het filter for target samples
TARGET_HET_LO = 0.20
TARGET_HET_HI = 0.80

# Significance threshold
P_THRESHOLD = 0.05

# Minimum phased het SNPs required
MIN_HETS = 3

# PON configuration - the deposited control BAF files
CNV_DEPOSIT_DIR = 'Data_files/mCA_calling/Real_data/EGA_deposit_CNV_BAF_LRR'

# Read PON sample list from master matrix
_pon_matrix = pd.read_csv(
    'Data_files/mCA_calling/PON_master_matrix.tsv',
    sep='\t', nrows=0
)
FINAL_TIMEPOINT_CONTROLS = [c for c in _pon_matrix.columns if c.startswith('CNTRL')]

# Known germline mCA chromosomes — skip these
KNOWN_GERMLINE_CHROMS = {
    'CNTRL_160_s8': ['chr7'],
    'CNTRL_169_s7': ['chr18'],
    'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
    'CNTRL_177_s4': ['chrX'],
    'CNTRL_181_s7': ['chr16'],
    'CNTRL_182_s4': ['chr2', 'chr3'],
    'CNTRL_183_s6': ['chr1'],
    'CNTRL_186_s4': ['chr17'],
    'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
    'CNTRL_193_s2': ['chr9', 'chrX'],
    'CNTRL_199_s7': ['chr5'],
    'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
    'CNTRL_164_s6': ['chrX'],
    'CNTRL_167_s2': ['chr3', 'chr8'],
    'CNTRL_171_s2': ['chr2'],
    'CNTRL_172_s7': ['chr7'],
    'CNTRL_175_s3': ['chr21'],
    'CNTRL_180_s3': ['chr14'],
    'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
    'CNTRL_190_s4': ['chrX'],
    'CNTRL_191_s7': ['chr4', 'chr14'],
    'CNTRL_194_s8': ['chr6'],
    'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
    'CNTRL_002_s8': ['chr1', 'chr8'],
    'CNTRL_003_s9': ['chr1', 'chr10'],
    'CNTRL_004_s10': ['chr19', 'chrX'],
    'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
    'CNTRL_162_s5': ['chr19', 'chrX'],
    'CNTRL_185_s4': ['chr15'],
    'CNTRL_189_s4': ['chr6', 'chr15'],
    'CNTRL_195_s3': ['chr3'],
    'CNTRL_196_s8': ['chr22'],
    'CNTRL_198_s2': ['chr3', 'chr6'],
}

DEBUG = True

# Permutation settings

In [54]:
def phased_dev_to_cf(pd_val, event):
    """Convert mean phased_dev to cell fraction estimate."""
    if pd_val <= 0:
        return 0.0
    if event in ('CN-LOH', 'CNLOH'):
        return float(np.clip(2 * pd_val, 0, 1))
    elif event == 'GAIN':
        return float(np.clip(4 * pd_val / (1 - 2 * pd_val) if pd_val < 0.5 else 1.0, 0, 1))
    elif event == 'LOSS':
        return float(np.clip(4 * pd_val / (1 + 2 * pd_val), 0, 1))
    else:
        return float(np.clip(2 * pd_val, 0, 1))


# Simulated samples: Using high CF sample phasing & coordinates to call mCA cell fraction in lower CF sample ('longitudinal analysis')

- Tests multiple reference CFs (100%, 75%, 50%, 25%) independently per family, showing how phasing quality degrades with lower reference CF.

For each simulated mCA family:
  1. Check if the mCA was detected (TP) by the unphased caller at a reference CF
  2. If yes, derive phasing from the reference sample's VAF (real-world workflow)
  3. Apply that phasing to all lower-CF samples in the family
  4. Test whether the mCA is detectable at each lower CF

Requires:
  - truth_comparison.csv (unphased caller results with detected coordinates)
  - simulation_manifest.csv
  - Simulated SNP files

### Configuration

In [55]:
# Reference CFs to test phasing from (highest priority first)
REFERENCE_CFS = [1.0, 0.75, 0.5, 0.25]

# Target CFs to test detection at
fractions = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]
TARGET_CFS = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]

# Minimum median AI in reference to accept phasing
MIN_MEDIAN_AI = 0.01


In [56]:
# PON configuration - the deposited control BAF files
CNV_DEPOSIT_DIR = 'Data_files/mCA_calling/Real_data/EGA_deposit_CNV_BAF_LRR'

# Read PON sample list from master matrix
_pon_matrix = pd.read_csv(
    'Data_files/mCA_calling/PON_master_matrix.tsv',
    sep='\t', nrows=0
)
FINAL_TIMEPOINT_CONTROLS = [c for c in _pon_matrix.columns if c.startswith('CNTRL')]

# Baseline weight (set after sweep)

# Known germline mCA chromosomes — skip these
KNOWN_GERMLINE_CHROMS = {
    'CNTRL_160_s8': ['chr7'],
    'CNTRL_169_s7': ['chr18'],
    'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
    'CNTRL_177_s4': ['chrX'],
    'CNTRL_181_s7': ['chr16'],
    'CNTRL_182_s4': ['chr2', 'chr3'],
    'CNTRL_183_s6': ['chr1'],
    'CNTRL_186_s4': ['chr17'],
    'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
    'CNTRL_193_s2': ['chr9', 'chrX'],
    'CNTRL_199_s7': ['chr5'],
    'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
    'CNTRL_164_s6': ['chrX'],
    'CNTRL_167_s2': ['chr3', 'chr8'],
    'CNTRL_171_s2': ['chr2'],
    'CNTRL_172_s7': ['chr7'],
    'CNTRL_175_s3': ['chr21'],
    'CNTRL_180_s3': ['chr14'],
    'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
    'CNTRL_190_s4': ['chrX'],
    'CNTRL_191_s7': ['chr4', 'chr14'],
    'CNTRL_194_s8': ['chr6'],
    'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
    'CNTRL_002_s8': ['chr1', 'chr8'],
    'CNTRL_003_s9': ['chr1', 'chr10'],
    'CNTRL_004_s10': ['chr19', 'chrX'],
    'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
    'CNTRL_162_s5': ['chr19', 'chrX'],
    'CNTRL_185_s4': ['chr15'],
    'CNTRL_189_s4': ['chr6', 'chr15'],
    'CNTRL_195_s3': ['chr3'],
    'CNTRL_196_s8': ['chr22'],
    'CNTRL_198_s2': ['chr3', 'chr6'],
}

### Core functions

In [57]:
# ═══════════════════════════════════════════════════════════════════════════
# SHARED FUNCTIONS — used by BOTH longitudinal pipeline and FPR pipeline
# ═══════════════════════════════════════════════════════════════════════════

def derive_phasing_from_vaf(snp_data, chrom, start, end, event_type=None,
                            min_hets=3, min_median_ai=0.01,
                            vaf_baselines=None):
    """
    Derive haplotype phasing from a high-CF sample's VAF.

    At high CF, het SNPs are pushed clearly above or below their baseline:
      VAF > baseline -> ALT on amplified haplotype -> phase = 1
      VAF < baseline -> ALT on other haplotype -> phase = 0

    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF columns
    chrom : str
    start, end : int - region coordinates
    event_type : str or None - 'CN-LOH', 'GAIN', 'LOSS'
    min_hets : int - minimum het SNPs required
    min_median_ai : float - minimum median AI to accept phasing
    vaf_baselines : dict {(chrom, position): median_vaf} or None
        Population-level baseline VAFs. If None, uses 0.5 for all positions.

    Returns
    -------
    phase_lookup : dict {position: phase} or None if phasing fails
    n_phased : int - number of phased SNPs
    median_ai : float - median allelic imbalance in reference
    """
    chrom_str = str(chrom)
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == chrom_str]
    region_snps = chrom_snps[
        (chrom_snps['position'] >= start) & (chrom_snps['position'] <= end)
    ]

    # VAF filter depends on event type
    # For CN-LOH/LOSS at high CF, original hets are pushed to VAF ~0 or ~1
    # (they look homozygous). We still need to phase them.
    if event_type in ('CN-LOH', 'CNLOH', 'LOSS'):
        ref_hets = region_snps.copy()
    else:
        ref_hets = region_snps[
            (region_snps['VAF'] >= 0.01) & (region_snps['VAF'] <= 0.99)
        ].copy()

    # deduplicate positions keeping highest AI row
    ref_hets = ref_hets.copy()
    ref_hets['_ai'] = np.abs(ref_hets['VAF'].astype(float) - 0.5)
    ref_hets = (ref_hets.sort_values('_ai', ascending=False)
                        .drop_duplicates(subset='position', keep='first')
                        .drop(columns='_ai'))

    if len(ref_hets) < min_hets:
        return None, len(ref_hets), 0

    # Get baseline for each position
    positions = ref_hets['position'].astype(int).values
    vafs = ref_hets['VAF'].astype(float).values

    if vaf_baselines is not None:
        bl = np.array([vaf_baselines.get((chrom_str, int(pos)), 0.5) for pos in positions])
    else:
        bl = np.full(len(positions), 0.5)

    # Check signal strength (AI relative to baseline)
    ai_values = np.abs(vafs - bl)
    median_ai = float(np.median(ai_values))
    if event_type not in ('CN-LOH', 'CNLOH', 'LOSS') and median_ai < min_median_ai:
        return None, len(ref_hets), median_ai

    # Derive phase relative to baseline
    phases = (vafs > bl).astype(float)
    phase_lookup = dict(zip(positions.tolist(), phases.tolist()))

    return phase_lookup, len(ref_hets), median_ai

def estimate_cf_phased(snp_data, chrom, start, end, phase_lookup, event_type,
                       het_lo=0.2, het_hi=0.8, min_hets=3, p_threshold=0.05,
                       vaf_baselines=None, n_permutations=10000, perm_seed=42,
                       baseline_weight=1.0):
    """
    Estimate cell fraction using externally-derived phasing.

    Computes phased_dev = (2*phase - 1) * (VAF - baseline) for each het SNP,
    then tests whether mean phased_dev is significantly > 0 using both a
    one-sided t-test and a sign-flip permutation test (both must yield
    P < p_threshold for the event to be called significant).

    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF columns
    chrom : str
    start, end : int - region coordinates
    phase_lookup : dict {position: phase} from reference sample
    event_type : str ('CN-LOH', 'GAIN', 'LOSS')
    het_lo, het_hi : float - VAF range for het filter
    min_hets : int - minimum phased het SNPs required
    p_threshold : float - significance threshold (applied to both tests)
    vaf_baselines : dict {(chrom, position): median_vaf} or None
        Population-level baseline VAFs. If None, uses 0.5 for all positions.
    n_permutations : int - number of sign-flip permutations
    perm_seed : int - random seed for permutation test

    Returns
    -------
    dict with cf_estimate, p_onesided, p_permutation, significant, n_hets,
         mean_phased_dev, t_stat
    or None if insufficient data
    """
    chrom_str = str(chrom)
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == chrom_str]

    if event_type in ('CN-LOH', 'CNLOH', 'LOSS'):
        
        def compute_devs(lo, hi):
            p_hets = chrom_snps[
                (chrom_snps['VAF'] >= lo) &
                (chrom_snps['VAF'] <= hi)
            ].copy()
            p_hets = p_hets[
                (p_hets['position'] >= start) &
                (p_hets['position'] <= end)
            ]
            devs = []
            for pos, vaf in zip(p_hets['position'].astype(int).values,
                                p_hets['VAF'].astype(float).values):
                if pos in phase_lookup:
                    phase = phase_lookup[pos]
                    bl = vaf_baselines.get((chrom_str, int(pos)), 0.5) if vaf_baselines else 0.5
                    bl_partial = 0.5 + baseline_weight * (bl - 0.5)
                    devs.append((2 * phase - 1) * (vaf - bl_partial))
            return devs
        
        region_size_mb = (end - start) / 1e6
        density_threshold = 0.5  # SNPs/Mb

        # Pass 1: 0.2-0.8
        devs = compute_devs(0.2, 0.8)
        pass_used = 1
        eff_het_lo_used, eff_het_hi_used = 0.2, 0.8
        density = len(devs) / region_size_mb

        # Pass 2: 0.05-0.95
        if density < density_threshold or len(devs) < min_hets:
            devs = compute_devs(0.05, 0.95)
            pass_used = 2
            eff_het_lo_used, eff_het_hi_used = 0.05, 0.95
            density = len(devs) / region_size_mb

        # Pass 3: 0.0-1.0
        if density < density_threshold or len(devs) < min_hets:
            devs = compute_devs(0.001, 0.999)
            pass_used = 3
            eff_het_lo_used, eff_het_hi_used = 0.001, 0.999

        # Set effective bounds to match whichever pass was used
        # (devs already computed — pass directly to phased_dev array)
        phased_dev = np.array(devs)
        n_hets = len(phased_dev)

        if n_hets < min_hets:
            return None

        # Skip the main het filter block below — already computed
        mean_dev = np.mean(phased_dev)
        t_stat, p_two = ttest_1samp(phased_dev, 0)
        p_onesided = p_two / 2 if t_stat > 0 else 1 - p_two / 2
        rng = np.random.default_rng(perm_seed)
        abs_vals = np.abs(phased_dev)
        null_means = np.array([
            np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
            for _ in range(n_permutations)
        ])
        p_permutation = float(np.mean(null_means >= mean_dev))
        significant = (p_onesided < p_threshold)
        cf_estimate = phased_dev_to_cf(mean_dev, event_type)

        se_dev = float(np.std(phased_dev, ddof=1)) / np.sqrt(n_hets)
        t_crit = t_dist.ppf(0.975, df=n_hets - 1)
        cf_lower_95 = phased_dev_to_cf(mean_dev - t_crit * se_dev, event_type)
        cf_upper_95 = phased_dev_to_cf(mean_dev + t_crit * se_dev, event_type)

        return {
            'cf_estimate': cf_estimate,
            'cf_lower_95': cf_lower_95,
            'cf_upper_95': cf_upper_95,
            'p_twosided': p_two,
            'p_onesided': p_onesided,
            'p_permutation': p_permutation,
            'significant': significant,
            'n_hets': n_hets,
            'mean_phased_dev': mean_dev,
            't_stat': t_stat,
            'pass_used': pass_used,
            'eff_het_lo': eff_het_lo_used,
            'eff_het_hi': eff_het_hi_used,
        }

    else:
        # GAIN — standard het filter, no multi-pass needed
        pass_used = 1
        eff_het_lo = het_lo
        eff_het_hi = het_hi
        eff_het_lo_used = het_lo
        eff_het_hi_used = het_hi

    hets = chrom_snps[
        (chrom_snps['VAF'] >= eff_het_lo) & (chrom_snps['VAF'] <= eff_het_hi)
    ].copy()

    if len(hets) == 0:
        return None

    # Filter to region + phased positions
    region_hets = hets[
        (hets['position'] >= start) & (hets['position'] <= end)
    ].copy()

    positions = region_hets['position'].astype(int).values
    vafs = region_hets['VAF'].astype(float).values

    phased_dev = []
    for pos, vaf in zip(positions, vafs):
        if pos in phase_lookup:
            phase = phase_lookup[pos]
            if vaf_baselines is not None:
                bl = vaf_baselines.get((chrom_str, int(pos)), 0.5)
            else:
                bl = 0.5
            bl_partial = 0.5 + baseline_weight * (bl-0.5)
            dev = (2 * phase - 1) * (vaf - bl_partial)
            phased_dev.append(dev)

    phased_dev = np.array(phased_dev)
    n_hets = len(phased_dev)

    if n_hets < min_hets:
        return None

    # --- One-sample t-test: is mean phased_dev > 0? ---
    mean_dev = np.mean(phased_dev)
    t_stat, p_two = ttest_1samp(phased_dev, 0)

    # One-sided p-value (we expect positive deviation)
    p_onesided = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    # --- Sign-flip permutation test ---
    rng = np.random.default_rng(perm_seed)
    abs_vals = np.abs(phased_dev)
    null_means = np.array([
        np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
        for _ in range(n_permutations)
    ])
    p_permutation = float(np.mean(null_means >= mean_dev))

    # --- Significance requires just t-test (as testing in known regions) ---
    # significant = (p_onesided < p_threshold) and (p_permutation < p_threshold)
    significant = (p_onesided < p_threshold)

    # CF estimation
    cf_estimate = phased_dev_to_cf(mean_dev, event_type)
    se_dev = float(np.std(phased_dev, ddof=1)) / np.sqrt(n_hets)
    t_crit = t_dist.ppf(0.975, df=n_hets - 1)
    cf_lower_95 = phased_dev_to_cf(mean_dev - t_crit * se_dev, event_type)
    cf_upper_95 = phased_dev_to_cf(mean_dev + t_crit * se_dev, event_type)
    return {
        'cf_estimate': cf_estimate,
        'cf_lower_95': cf_lower_95,
        'cf_upper_95': cf_upper_95,
        'p_twosided': p_two,
        'p_onesided': p_onesided,
        'p_permutation': p_permutation,
        'significant': significant,
        'n_hets': n_hets,
        'mean_phased_dev': mean_dev,
        't_stat': t_stat,
        'pass_used': pass_used,      # 1, 2, or 3
        'eff_het_lo': eff_het_lo_used,
        'eff_het_hi': eff_het_hi_used,
    }

In [ ]:
def build_pon_snp_file_map(deposit_dir, allowed_samples=None):
    """
    Locate the per-SNP BAF file for each PON control in the deposited CNV BAF/LRR set.

    The deposit one file per sample-timepoint - and the filenames are not uniform
    (some carry the library UDI suffix, some do not), so files are matched by sample prefix
    rather than by a constructed path. All 36 PON controls are present in the deposit.

    Returns dict {sample_name: snp_filepath}
    """
    SUFFIX = 'variant_calling_only_SNPs_annovar_annotated.txt'
    if not os.path.isdir(deposit_dir):
        print(f"  Warning: deposit directory not found: {deposit_dir}")
        return {}

    files = [f for f in os.listdir(deposit_dir) if f.endswith(SUFFIX)]
    pon_snp_files = {}
    for sample in (allowed_samples or []):
        prefix = (sample + '_').lower()
        match = sorted(f for f in files if f.lower().startswith(prefix))
        if match:
            pon_snp_files[sample] = os.path.join(deposit_dir, match[0])
        else:
            print(f"  Warning: no SNP file found for {sample}")

    print(f"  Found SNP files for {len(pon_snp_files)} PON samples")
    return pon_snp_files

### Longitudinal phasing pipeline

In [59]:
def get_sample_filename(manifest, control, event_type, fraction, chrom, start, end):
    """Find the SNP filename for a specific sample in the manifest."""
    type_map = {'CN-LOH': 'CNLOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    sim_type = type_map.get(event_type, event_type)
    
    matches = manifest[
        (manifest['Sample'] == control) &
        (manifest['Type'] == sim_type) &
        (np.isclose(manifest['Fraction_Percent'], fraction)) &
        (manifest['Chromosome'] == chrom) &
        (manifest['Start_bp'] == start) &
        (manifest['End_bp'] == end)
    ]
    
    if len(matches) == 0:
        return None
    
    # Derive SNP filename from the LRR filename
    lrr_file = matches.iloc[0]['File_Name']
    snp_file = lrr_file.replace('_PON_normalised_read_depths_and_LRR.txt', '_SNPs.txt')
    return snp_file

In [60]:
def run_longitudinal_pipeline(truth_csv, manifest_csv, simulated_dir,
                              reference_cfs=None, target_cfs=None,
                              het_lo=0.02, het_hi=0.98, min_hets=3,
                              min_median_ai=0.01, p_threshold=0.05,
                              vaf_baselines=None,
                              baseline_weight=1.0,
                              debug=True):
    """
    Run the full longitudinal phasing analysis.
    
    For each family × reference_cf:
      - Check if mCA was TP at reference_cf (from unphased caller)
      - If yes, derive phasing from reference sample's VAF
      - Apply phasing to all lower-CF samples
      - Record detection results
    
    Parameters
    ----------
    truth_csv : str - path to truth_comparison.csv
    manifest_csv : str - path to simulation_manifest.csv
    simulated_dir : str - path to simulated sample files
    reference_cfs : list - CF levels to try as phasing reference
    target_cfs : list - CF levels to test detection at
    het_lo, het_hi : float - VAF range for het filter
    min_hets : int - minimum phased het SNPs required
    min_median_ai : float - minimum median AI to accept phasing
    p_threshold : float - significance threshold
    debug : bool - print progress
    """
    if reference_cfs is None:
        reference_cfs = [1.0, 0.75, 0.5, 0.25]
    if target_cfs is None:
        target_cfs = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 1.00]
    
    # Load data
    tc = pd.read_csv(truth_csv)
    manifest = pd.read_csv(manifest_csv)
    
    event_map = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    tc['event'] = tc['Truth_Type'].map(event_map)
    
    # Assign size categories
    tc['size'] = pd.cut(
        tc['Truth_Length_Mb'],
        bins=[0, 2.5, 3.5, 5.5, 10.5, 20.5, 200],
        labels=['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']
    )
    
    # Group into families
    family_cols = ['Sample', 'Truth_Type', 'Truth_Chromosome', 'Truth_Start', 'Truth_End']
    families = tc.groupby(family_cols)
    
    n_families = families.ngroups
    print(f"Total families: {n_families}")
    
    all_results = []
    
    # ══════════════════════════════════════════════════════════════════════
    # DEBUG: Track skip reasons by event type and reference CF
    # ══════════════════════════════════════════════════════════════════════
    from collections import defaultdict
    skip_counts = defaultdict(lambda: defaultdict(int))
    # Keys: (event, ref_cf) -> reason -> count
    proceed_counts = defaultdict(lambda: defaultdict(int))
    
    for fam_idx, (fam_key, fam_df) in enumerate(families):
        control, truth_type, chrom, true_start, true_end = fam_key
        event = event_map[truth_type]
        
        # Get metadata from first row
        first = fam_df.iloc[0]
        size = first['size']
        geometry = first['Truth_Geometry']
        
        if fam_idx % 100 == 0 and debug:
            print(f"  Processing family {fam_idx+1}/{n_families}...")
        
        # ── Try each reference CF independently ──
        for ref_cf in reference_cfs:
            
            debug_key = (event, ref_cf)
            
            # Was this mCA detected at this reference CF?
            ref_row = fam_df[np.isclose(fam_df['Truth_Fraction'], ref_cf)]
            if len(ref_row) == 0:
                skip_counts[debug_key]['no_ref_row_at_cf'] += 1
                continue
            
            ref_row = ref_row.iloc[0]
            
            if ref_row['Status'] != 'TRUE_POSITIVE':
                skip_counts[debug_key]['not_true_positive'] += 1
                continue
            
            # Get DETECTED coordinates (not ground truth)
            det_start = int(ref_row['Detected_Start'])
            det_end = int(ref_row['Detected_End'])
            det_type = ref_row['Detected_Type']
            
            # Find and load the reference SNP file
            ref_snp_name = get_sample_filename(
                manifest, control, event, ref_cf, chrom, true_start, true_end
            )
            if ref_snp_name is None:
                skip_counts[debug_key]['get_sample_filename_returned_None'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: get_sample_filename returned None "
                          f"for {control} {chrom}:{true_start}-{true_end}")
                continue
            
            ref_snp_path = os.path.join(simulated_dir, ref_snp_name)
            if not os.path.exists(ref_snp_path):
                skip_counts[debug_key]['ref_file_not_found'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: ref SNP file not found: {ref_snp_path}")
                continue
            
            try:
                ref_snps = pd.read_csv(ref_snp_path, sep='\t')
            except Exception as e:
                skip_counts[debug_key]['ref_file_read_error'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: error reading ref file: {e}")
                continue
            
            # Derive phasing from reference VAF at DETECTED coordinates
            phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
                ref_snps, chrom, det_start, det_end, event_type=event,
                min_hets=min_hets, min_median_ai=min_median_ai,
                vaf_baselines=vaf_baselines
            )
            
            if phase_lookup is None:
                skip_counts[debug_key]['phasing_failed'] += 1
                if debug:
                    print(f"    DEBUG SKIP [{event}] ref={ref_cf}: derive_phasing_from_vaf returned None "
                          f"for {control} {chrom}:{det_start}-{det_end} "
                          f"(n_phased={n_phased}, median_ai={median_ai:.4f})")
                continue
            
            # If we get here, phasing succeeded — count it
            proceed_counts[debug_key]['phasing_succeeded'] += 1
            
            # ── Apply phasing to all target CFs ──
            for target_cf in target_cfs:
                if target_cf >= ref_cf:
                    continue  # Only test lower CFs
                
                target_debug_key = (event, ref_cf, target_cf)
                
                # Find and load target SNP file
                target_snp_name = get_sample_filename(
                    manifest, control, event, target_cf, chrom, true_start, true_end
                )
                if target_snp_name is None:
                    skip_counts[debug_key]['target_filename_None'] += 1
                    continue
                
                target_snp_path = os.path.join(simulated_dir, target_snp_name)
                if not os.path.exists(target_snp_path):
                    skip_counts[debug_key]['target_file_not_found'] += 1
                    continue
                
                try:
                    target_snps = pd.read_csv(target_snp_path, sep='\t')
                except Exception:
                    skip_counts[debug_key]['target_file_read_error'] += 1
                    continue
                
                # Estimate CF using reference-derived phasing at DETECTED coordinates
                result = estimate_cf_phased(
                    target_snps, chrom, det_start, det_end, phase_lookup, event,
                    het_lo=het_lo, het_hi=het_hi, min_hets=min_hets,
                    p_threshold=p_threshold,
                    vaf_baselines=vaf_baselines, baseline_weight=baseline_weight
                )
                
                if result is None:
                    all_results.append({
                        'control': control,
                        'event': event,
                        'chromosome': chrom,
                        'true_start': true_start,
                        'true_end': true_end,
                        'det_start': det_start,
                        'det_end': det_end,
                        'size': size,
                        'geometry': geometry,
                        'ref_cf': ref_cf,
                        'target_cf': target_cf,
                        'ref_n_phased': n_phased,
                        'ref_median_ai': median_ai,
                        'significant': False,
                        'cf_estimate': np.nan,
                        'p_onesided': np.nan,
                        'p_twosided': np.nan,
                        'n_hets': 0,
                        'mean_phased_dev': np.nan,
                        't_stat': np.nan,
                        'status': 'insufficient_hets',
                    })
                    continue
                
                all_results.append({
                    'control': control,
                    'event': event,
                    'chromosome': chrom,
                    'true_start': true_start,
                    'true_end': true_end,
                    'det_start': det_start,
                    'det_end': det_end,
                    'size': size,
                    'geometry': geometry,
                    'ref_cf': ref_cf,
                    'target_cf': target_cf,
                    'ref_n_phased': n_phased,
                    'ref_median_ai': median_ai,
                    'significant': result['significant'],
                    'cf_estimate': result['cf_estimate'],
                    'p_onesided': result['p_onesided'],
                    'p_twosided': result['p_twosided'],
                    'n_hets': result['n_hets'],
                    'mean_phased_dev': result['mean_phased_dev'],
                    't_stat': result['t_stat'],
                    'status': 'tested',
                })
    
    # ══════════════════════════════════════════════════════════════════════
    # DEBUG: Print skip reason summary
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "="*80)
    print("DEBUG: FAMILY SKIP REASONS BY EVENT TYPE AND REFERENCE CF")
    print("="*80)
    
    for event in ['CN-LOH', 'GAIN', 'LOSS']:
        print(f"\n  {event}:")
        for ref_cf in (reference_cfs or [1.0, 0.75, 0.5, 0.25]):
            key = (event, ref_cf)
            if key in skip_counts or key in proceed_counts:
                print(f"    ref={ref_cf*100:.0f}% CF:")
                for reason, count in sorted(skip_counts[key].items()):
                    print(f"      SKIPPED - {reason}: {count}")
                for reason, count in sorted(proceed_counts[key].items()):
                    print(f"      OK      - {reason}: {count}")
    
    print("\n" + "="*80)
    print("DEBUG: SUMMARY — families reaching phasing step vs dropped before")
    print("="*80)
    for event in ['CN-LOH', 'GAIN', 'LOSS']:
        for ref_cf in (reference_cfs or [1.0, 0.75, 0.5, 0.25]):
            key = (event, ref_cf)
            succeeded = proceed_counts[key].get('phasing_succeeded', 0)
            total_skipped = sum(skip_counts[key].values())
            # Subtract target-level skips (those happen after phasing succeeded)
            ref_level_skips = sum(v for k, v in skip_counts[key].items() 
                                  if not k.startswith('target_'))
            print(f"  {event:8s} ref={ref_cf*100:>3.0f}%: "
                  f"{succeeded} phased OK, "
                  f"{ref_level_skips} dropped before phasing "
                  f"(breakdown: {dict([(k,v) for k,v in skip_counts[key].items() if not k.startswith('target_')])})")
    
    results_df = pd.DataFrame(all_results)
    return results_df

## Assessing false positive rate

In [61]:
# ═══════════════════════════════════════════════════════════════════════════
# FALSE POSITIVE RATE (FPR) ANALYSIS — within-family null design
# ═══════════════════════════════════════════════════════════════════════════
#
# For each simulated family in which the mCA was detected at the index cell
# fraction, phasing is derived from that index sample and applied to the
# ORIGINAL unmodified control file for the same individual (0% cell fraction).
# Any significant detection there is a false positive.
#
# This mirrors the cohort analysis exactly — phase from a high-cell-fraction
# timepoint, then test an earlier timepoint from the same person — so the false
# positives it measures are the ones that matter: those arising from phasing
# noise rather than from cross-individual BAF differences.

# ── Known germline events to exclude ──
KNOWN_EVENT_CHROMS = {
    'CNTRL_160_s8': ['chr7'],
    'CNTRL_169_s7': ['chr18'],
    'CNTRL_174_s3': ['chr4', 'chr7', 'chr13', 'chrX'],
    'CNTRL_177_s4': ['chrX'],
    'CNTRL_181_s7': ['chr16'],
    'CNTRL_182_s4': ['chr2', 'chr3'],
    'CNTRL_183_s6': ['chr1'],
    'CNTRL_186_s4': ['chr17'],
    'CNTRL_188_s7': ['chr7', 'chr17', 'chrX'],
    'CNTRL_193_s2': ['chr9', 'chrX'],
    'CNTRL_199_s7': ['chr5'],
    'CNTRL_163_s6': ['chr5', 'chr16', 'chrX'],
    'CNTRL_164_s6': ['chrX'],
    'CNTRL_167_s2': ['chr3', 'chr8'],
    'CNTRL_171_s2': ['chr2'],
    'CNTRL_172_s7': ['chr7'],
    'CNTRL_175_s3': ['chr21'],
    'CNTRL_180_s3': ['chr14'],
    'CNTRL_187_s2': ['chr10', 'chr17', 'chrX'],
    'CNTRL_190_s4': ['chrX'],
    'CNTRL_191_s7': ['chr4', 'chr14'],
    'CNTRL_194_s8': ['chr6'],
    'CNTRL_001_s10': ['chr2', 'chr13', 'chrX'],
    'CNTRL_002_s8': ['chr1', 'chr8'],
    'CNTRL_003_s9': ['chr1', 'chr10'],
    'CNTRL_004_s10': ['chr19', 'chrX'],
    'CNTRL_005_s9': ['chr6', 'chr9', 'chrX'],
    'CNTRL_162_s5': ['chr19', 'chrX'],
    'CNTRL_185_s4': ['chr15'],
    'CNTRL_189_s4': ['chr6', 'chr15'],
    'CNTRL_195_s3': ['chr3'],
    'CNTRL_196_s8': ['chr22'],
    'CNTRL_198_s2': ['chr3', 'chr6'],
}

# ── Panel BED file ──
PANEL_BED_FILE = "Data_files/TWIST_CNV_panel_TE-95031423_h19.bed"
MIN_PROBES_FOR_REGION = 20  # Minimum probes in a null region
NULL_REGIONS_PER_FAMILY = 5  # Number of null regions to test per family


## Within-family FPR analysis

In [62]:
def run_within_family_fpr_pipeline(
        truth_csv, manifest_csv, simulated_dir,
        original_snp_files,
        reference_cf=0.5,
        het_lo=0.02, het_hi=0.98,
        min_hets=MIN_HETS,
        min_median_ai=MIN_MEDIAN_AI,
        p_threshold=P_THRESHOLD,
        vaf_baselines=None,
        baseline_weight=1.0,
        known_germline_chroms=None,
        debug=True):
    
    """
    Within-family FPR analysis.

    For each family where the mCA was detected at reference_cf:
      1. Derive phasing from the simulated reference_cf sample (same individual)
      2. Test the original unmodified control SNP file (0% CF) in the same region
      3. Any significant result = false positive

    This mirrors the exact clinical scenario: phasing from a high-CF timepoint,
    testing whether the mCA is detectable in an earlier null timepoint.
    No cross-individual VAF differences, so baseline correction is not needed.
    """

    if known_germline_chroms is None:
        known_germline_chroms = {}

    tc = pd.read_csv(truth_csv)
    manifest = pd.read_csv(manifest_csv)

    event_map = {'CNLOH': 'CN-LOH', 'GAIN': 'GAIN', 'LOSS': 'LOSS'}
    tc['event'] = tc['Truth_Type'].map(event_map)
    tc['size'] = pd.cut(
        tc['Truth_Length_Mb'],
        bins=[0, 2.5, 3.5, 5.5, 10.5, 20.5, 200],
        labels=['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']
    )

    # ── Group families ──
    family_cols = ['Sample', 'Truth_Type', 'Truth_Chromosome', 'Truth_Start', 'Truth_End']
    families = tc.groupby(family_cols)
    n_families = families.ngroups
    print(f"Total families: {n_families}")
    print(f"Reference CF for phasing: {reference_cf*100:.0f}%")
    print(f"Null target: original unmodified control (0% CF)")

    all_results = []
    stats = defaultdict(int)

    for fam_idx, (fam_key, fam_df) in enumerate(families):
        control, truth_type, chrom, true_start, true_end = fam_key
        event = event_map[truth_type]
        first = fam_df.iloc[0]
        size = first['size']
        geometry = first['Truth_Geometry']

        if fam_idx % 100 == 0 and debug:
            print(f"  FPR: family {fam_idx+1}/{n_families} "
                  f"({event}, {size}, {chrom}, {control})...")

        # ── Check mCA was detected at reference CF ──
        ref_row = fam_df[np.isclose(fam_df['Truth_Fraction'], reference_cf)]
        if len(ref_row) == 0:
            stats['no_ref_row'] += 1
            continue
        ref_row = ref_row.iloc[0]
        if ref_row['Status'] != 'TRUE_POSITIVE':
            stats['ref_not_tp'] += 1
            continue

        # ── Skip known germline mCA chromosomes ──
        germline_chroms = set(known_germline_chroms.get(control, []))
        if chrom in germline_chroms:
            stats['skipped_germline'] += 1
            continue

        # ── Load simulated reference SNP file ──
        ref_snp_name = get_sample_filename(
            manifest, control, event, reference_cf, chrom, true_start, true_end
        )
        if ref_snp_name is None:
            stats['ref_file_not_found'] += 1
            continue
        ref_snp_path = os.path.join(simulated_dir, ref_snp_name)
        if not os.path.exists(ref_snp_path):
            stats['ref_file_not_found'] += 1
            continue
        try:
            ref_snps = pd.read_csv(ref_snp_path, sep='\t')
        except Exception as e:
            stats['ref_file_not_found'] += 1
            continue

        # ── Find original unmodified control SNP file (0% CF) ──
        if control not in original_snp_files:
            stats['original_not_found'] += 1
            continue
        try:
            original_snps = pd.read_csv(original_snp_files[control], sep='\t')
        except Exception as e:
            stats['original_not_found'] += 1
            continue

        # ── Derive phasing from simulated reference sample ──
        phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
            ref_snps, chrom, true_start, true_end,
            event_type=event,
            min_hets=min_hets, min_median_ai=min_median_ai,
            vaf_baselines=vaf_baselines
        )

        if phase_lookup is None:
            stats['phasing_failed'] += 1
            continue

        stats['families_tested'] += 1

        # ── Test original control in same region ──
        result = estimate_cf_phased(
            original_snps, chrom, true_start, true_end,
            phase_lookup, event,
            het_lo=het_lo, het_hi=het_hi, min_hets=min_hets,
            p_threshold=p_threshold,
            vaf_baselines=vaf_baselines,
            baseline_weight=baseline_weight
        )

        if result is None:
            all_results.append({
                'control': control, 'event': event,
                'chromosome': chrom, 'start': true_start, 'end': true_end,
                'size': size, 'geometry': geometry,
                'ref_cf': reference_cf, 'n_phased_ref': n_phased,
                'median_ai_ref': median_ai,
                'significant': False, 'cf_estimate': np.nan,
                'p_onesided': np.nan, 'p_twosided': np.nan,
                'p_permutation': np.nan, 'n_hets': 0,
                'mean_phased_dev': np.nan, 't_stat': np.nan,
                'status': 'insufficient_hets',
            })
        else:
            all_results.append({
                'control': control, 'event': event,
                'chromosome': chrom, 'start': true_start, 'end': true_end,
                'size': size, 'geometry': geometry,
                'ref_cf': reference_cf, 'n_phased_ref': n_phased,
                'median_ai_ref': median_ai,
                'significant': result['significant'],
                'cf_estimate': result['cf_estimate'],
                'p_onesided': result['p_onesided'],
                'p_twosided': result['p_twosided'],
                'p_permutation': result['p_permutation'],
                'n_hets': result['n_hets'],
                'mean_phased_dev': result['mean_phased_dev'],
                't_stat': result['t_stat'],
                'status': 'tested',
            })

            if result['significant']:
                stats['false_positives'] += 1
            stats['null_tests'] += 1

    fpr_results = pd.DataFrame(all_results)

    # ── Summary ──
    print("\n" + "=" * 80)
    print("WITHIN-FAMILY FPR SUMMARY")
    print("=" * 80)
    print(f"  Families tested:        {stats['families_tested']}")
    print(f"  Phasing failed:         {stats['phasing_failed']}")
    print(f"  Original not found:     {stats['original_not_found']}")
    print(f"  Skipped germline:       {stats['skipped_germline']}")
    print(f"  Total null tests:       {stats['null_tests']}")
    print(f"  False positives:        {stats['false_positives']}")
    if stats['null_tests'] > 0:
        overall_fpr = stats['false_positives'] / stats['null_tests'] * 100
        print(f"  Overall FPR:            {overall_fpr:.2f}%")
        print(f"  Expected under null:    ~5% (alpha = 0.05)")

    tested = fpr_results[fpr_results['status'] == 'tested']
    if len(tested) > 0:
        print(f"\n  FPR by event type:")
        for evt in ['CN-LOH', 'GAIN', 'LOSS']:
            evt_data = tested[tested['event'] == evt]
            if len(evt_data) == 0:
                continue
            n = len(evt_data)
            n_fp = int(evt_data['significant'].sum())
            print(f"    {evt}: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

        print(f"\n  FPR by event size:")
        for sz in ['2Mb', '3Mb', '5Mb', '10Mb', '20Mb', 'WholeArm']:
            sz_data = tested[tested['size'] == sz]
            if len(sz_data) == 0:
                continue
            n = len(sz_data)
            n_fp = int(sz_data['significant'].sum())
            print(f"    {sz}: {n_fp}/{n} ({n_fp/n*100:.2f}%)")

    return fpr_results

In [63]:
# Find original (unmodified) control SNP files — used as 0% CF null targets
original_snp_files = build_pon_snp_file_map(
    CNV_DEPOSIT_DIR,
    allowed_samples=FINAL_TIMEPOINT_CONTROLS
)

all_fpr_results = []
for ref_cf in [0.25, 0.5, 0.75, 1.0]:
    results = run_within_family_fpr_pipeline(
        truth_csv=TRUTH_CSV,
        manifest_csv=MANIFEST_CSV,
        simulated_dir=SIMULATED_DIR,
        original_snp_files=original_snp_files,
        reference_cf=ref_cf,
        het_lo=TARGET_HET_LO,
        het_hi=TARGET_HET_HI,
        min_hets=MIN_HETS,
        min_median_ai=MIN_MEDIAN_AI,
        p_threshold=P_THRESHOLD,
        vaf_baselines=None,
        baseline_weight=1.0,
        known_germline_chroms=KNOWN_GERMLINE_CHROMS,
        debug=True
    )
    all_fpr_results.append(results)

# Save results
os.makedirs(OUTPUT_DIR, exist_ok=True)   #create the output folder on first run
within_fpr_path = os.path.join(OUTPUT_DIR, 'Within_family_fpr_results.csv')
within_fpr_results = pd.concat(all_fpr_results, ignore_index=True)
within_fpr_results.to_csv(within_fpr_path, index=False)
print(f"\n✅ Saved {len(within_fpr_results)} results to {within_fpr_path}")

  Found SNP files for 36 PON samples
Total families: 879
Reference CF for phasing: 25%
Null target: original unmodified control (0% CF)
  FPR: family 1/879 (CN-LOH, 2Mb, chr10, CNTRL_004_s10)...
  FPR: family 101/879 (CN-LOH, 5Mb, chr10, CNTRL_005_s9)...
  FPR: family 201/879 (CN-LOH, 10Mb, chr12, CNTRL_171_s2)...
  FPR: family 301/879 (CN-LOH, 2Mb, chr1, CNTRL_173_s3)...
  FPR: family 401/879 (CN-LOH, 10Mb, chr14, CNTRL_186_s4)...
  FPR: family 501/879 (CN-LOH, 5Mb, chr20, CNTRL_194_s8)...
  FPR: family 601/879 (CN-LOH, 2Mb, chr4, CNTRL_198_s2)...
  FPR: family 701/879 (CN-LOH, 5Mb, chr2, CNTRL_200_s2)...
  FPR: family 801/879 (CN-LOH, 20Mb, chr18, CNTRL_204_s5)...

WITHIN-FAMILY FPR SUMMARY
  Families tested:        326
  Phasing failed:         0
  Original not found:     0
  Skipped germline:       0
  Total null tests:       326
  False positives:        27
  Overall FPR:            8.28%
  Expected under null:    ~5% (alpha = 0.05)

  FPR by event type:
    CN-LOH: 10/121 (8.26%)

In [64]:
sens_results = run_longitudinal_pipeline(
    truth_csv=TRUTH_CSV,
    manifest_csv=MANIFEST_CSV,
    simulated_dir=SIMULATED_DIR,
    reference_cfs=[0.25, 0.5, 0.75, 1.0],
    target_cfs=TARGET_CFS,
    het_lo=TARGET_HET_LO,
    het_hi=TARGET_HET_HI,
    min_hets=MIN_HETS,
    min_median_ai=MIN_MEDIAN_AI,
    p_threshold=P_THRESHOLD,
    vaf_baselines=None,
    baseline_weight=1.0,
    debug=True
)

report_cfs = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.10, 0.20, 0.25, 0.5, 0.75]

for ref_cf in [0.25, 0.5, 0.75, 1.0]:
    print(f"\n{'='*80}")
    print(f"SENSITIVITY SUMMARY (ref={ref_cf*100:.0f}%)")
    print('='*80)
    tested = sens_results[np.isclose(sens_results['ref_cf'], ref_cf)]

    print(f"\n  Overall:")
    for cf in report_cfs:
        subset = tested[np.isclose(tested['target_cf'], cf)]
        if len(subset) == 0:
            continue
        detected = int(subset['significant'].sum())
        total = len(subset)
        print(f"    {cf*100:>5.1f}% CF: {detected}/{total} ({100*detected/total:.1f}%)")

    for evt in ['CN-LOH', 'GAIN', 'LOSS']:
        print(f"\n  {evt}:")
        evt_data = tested[tested['event'] == evt]
        for cf in report_cfs:
            subset = evt_data[np.isclose(evt_data['target_cf'], cf)]
            if len(subset) == 0:
                continue
            detected = int(subset['significant'].sum())
            total = len(subset)
            print(f"    {cf*100:>5.1f}% CF: {detected}/{total} ({100*detected/total:.1f}%)")

# Save
os.makedirs(OUTPUT_DIR, exist_ok=True)   #create the output folder on first run
sens_output = os.path.join(OUTPUT_DIR, 'sensitivity_results_all_ref_cfs.csv')
sens_results.to_csv(sens_output, index=False)
print(f"\n✅ Saved to {sens_output}")

Total families: 879
  Processing family 1/879...
  Processing family 101/879...
  Processing family 201/879...
  Processing family 301/879...
  Processing family 401/879...
  Processing family 501/879...
  Processing family 601/879...
  Processing family 701/879...
  Processing family 801/879...

DEBUG: FAMILY SKIP REASONS BY EVENT TYPE AND REFERENCE CF

  CN-LOH:
    ref=25% CF:
      SKIPPED - not_true_positive: 172
      OK      - phasing_succeeded: 121
    ref=50% CF:
      SKIPPED - not_true_positive: 151
      OK      - phasing_succeeded: 142
    ref=75% CF:
      SKIPPED - not_true_positive: 93
      OK      - phasing_succeeded: 200
    ref=100% CF:
      SKIPPED - not_true_positive: 92
      OK      - phasing_succeeded: 201

  GAIN:
    ref=25% CF:
      SKIPPED - not_true_positive: 197
      OK      - phasing_succeeded: 96
    ref=50% CF:
      SKIPPED - not_true_positive: 118
      OK      - phasing_succeeded: 175
    ref=75% CF:
      SKIPPED - not_true_positive: 99
      OK